In [0]:
%sql
USE CATALOG primeinsurance;
USE SCHEMA silver;

In [0]:
customers_bronze = "primeinsurance.bronze.customers"
sales_bronze = "primeinsurance.bronze.sales"
claims_bronze = "primeinsurance.bronze.claims"
cars_bronze = "primeinsurance.bronze.cars"
policy_bronze = "primeinsurance.bronze.policy"
customers_silver = "primeinsurance.silver.customers"
claims_silver = "primeinsurance.silver.claims"
sales_silver = "primeinsurance.silver.sales"
policy_silver = "primeinsurance.silver.policy"
cars_silver = "primeinsurance.silver.cars"
customers_quarantine_path = "primeinsurance.silver.customers_quarantine"
claims_quarantine_path = "primeinsurance.silver.claims_quarantine"
sales_quarantine_path = "primeinsurance.silver.sales_quarantine"
policy_quarantine_path = "primeinsurance.silver.policy_quarantine"
cars_quarantine_path = "primeinsurance.silver.cars_quarantine"

In [0]:
customers_df = spark.read.table(customers_bronze)
sales_df = spark.read.table(sales_bronze)
claims_df = spark.read.table(claims_bronze)
cars_df = spark.read.table(cars_bronze)
policy_df = spark.read.table(policy_bronze)

In [0]:
display(customers_df)

**Normalization of tables**

In [0]:
from pyspark.sql.functions import col

def normalize_columns(df):
    return df.select([
        col(c).alias(c.lower().replace(" ", "_"))
        for c in df.columns
    ])

n_customers_df = normalize_columns(customers_df)
n_sales_df     = normalize_columns(sales_df)
n_claims_df    = normalize_columns(claims_df)
n_cars_df      = normalize_columns(cars_df)
n_policy_df    = normalize_columns(policy_df)

**Harmonization and Merging**

In [0]:
# Assigning all columns in customers tables
customer_cols = [
    "customer_id",
    "cust_id",
    "customerid",
    "custid", 
    "customers_id",
    "customersid",
]

region_cols = [
    "region",
    "reg",
]
state_cols = [
    "state",
    "st",
    "state of country"
]
city_cols = [
    "city",
    "city_in_state",
    "cityinstate",
    "cityofstate",
    "city_of_state",
]

job_cols = [
    "job",
    "occupation",
    "employment",
    "work"
]
marital_cols = [
    "marital",
    "marital_status",
    "maritalstatus",
]
education_cols = [
    "education",
    "edu"
    "educ"
    "graduation"
    "schooling"
]

default_cols = [
    "default",
    "def",
]
balance_cols = [
    "balance",
    "bal",
]
insurance_cols = [
    "hhinsurance",
    "insurance",
    "ins",
]
loan_cols = [
    "carloan",
    "loan",
    "loans",
]

timestamp_cols = [
    "load_timestamp",
    "loadtimestamp",
    "time_stamp",
    "timestamp",
    "ingest_timestamp",
    "ingesttimestamp",
]

source_cols = [
    "source",
    "source_file",
    "source_path",
    "source_location"
]

In [0]:
from pyspark.sql.functions import coalesce, col

def merge_columns(df, new_col, col_list):
    existing_cols = [c.lower() for c in col_list if c.lower() in df.columns]

    if not existing_cols:
        return df

    return df.withColumn(
        new_col,
        coalesce(*[col(c) for c in existing_cols])
    )

# Apply columns name transformations # 
# customers table
n_customers_df = merge_columns(n_customers_df, "customer_id", customer_cols)
n_customers_df = merge_columns(n_customers_df, "region", region_cols)
n_customers_df = merge_columns(n_customers_df, "state", state_cols)
n_customers_df = merge_columns(n_customers_df, "city", city_cols)
n_customers_df = merge_columns(n_customers_df, "job", job_cols)
n_customers_df = merge_columns(n_customers_df, "marital_status", marital_cols)
n_customers_df = merge_columns(n_customers_df, "education", education_cols)
n_customers_df = merge_columns(n_customers_df, "default", default_cols)
n_customers_df = merge_columns(n_customers_df, "balance", balance_cols)
n_customers_df = merge_columns(n_customers_df, "hh_insurance", insurance_cols)
n_customers_df = merge_columns(n_customers_df, "car_loan", loan_cols)
n_customers_df = merge_columns(n_customers_df, "load_timestamp", timestamp_cols)
n_customers_df = merge_columns(n_customers_df, "source_file", source_cols)

# claims table
n_claims_df = n_claims_df.withColumnRenamed("claimid", "claim_id")
n_claims_df = n_claims_df.withColumnRenamed("policyid", "policy_id")

# policy table
n_policy_df = n_policy_df.withColumnRenamed("policy_number", "policy_id")

# Final standardized column names of customers
final_cols = [
    "customer_id", "region", "state", "city", "job",
    "marital_status", "education", "default",
    "balance", "hh_insurance", "car_loan",
    "load_timestamp", "source_file"
]

# Retrieve only correct customers columns
h_customers_df = n_customers_df.select(*[c for c in n_customers_df.columns if c in final_cols])

In [0]:
display(n_claims_df)

In [0]:
display(h_customers_df)

**Expanding region names**

In [0]:
from pyspark.sql.functions import col, when, upper

r_customers_df = h_customers_df.withColumn(
    "region",
    when(col("region") == "C", "Central")
    .when(col("region") == "N", "North")
    .when(col("region") == "S", "South")
    .when(col("region") == "E", "East")
    .when(col("region") == "W", "West")
    .otherwise(col("region"))
)

In [0]:
display(r_customers_df)

**Minimizing source path**

In [0]:
from pyspark.sql.functions import col, regexp_extract

def extract_source_file(df):
    return df.withColumn(
        "source_file",
        regexp_extract(col("source_file"), r'([^/]+/[^/]+$|[^/]+$)', 1)
    )

s_customers_df = extract_source_file(r_customers_df)
s_claims_df   = extract_source_file(n_claims_df)
s_policy_df   = extract_source_file(n_policy_df)
s_sales_df    = extract_source_file(n_sales_df)
s_cars_df     = extract_source_file(n_cars_df)

In [0]:
display(s_claims_df)

**Date formatting in claims**

In [0]:
from pyspark.sql.functions import col, expr, current_timestamp, rand

# Safe casting
schema_map = {
    "incident_date": "TIMESTAMP",
    "claim_logged_On": "TIMESTAMP",
    "claim_processed_On": "TIMESTAMP",
}

def apply_try_cast(df, schema_map):
    for col_name, data_type in schema_map.items():
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                expr(f"try_cast({col_name} as {data_type})")
            )
    return df

d_claims_df = apply_try_cast(s_claims_df, schema_map)

# Correct timestamp generation
t_claims_df = (
    d_claims_df
    
    # incident_date → last 60 days
    .withColumn(
        "incident_date",
        expr("""
            timestamp_seconds(
                unix_timestamp(current_timestamp()) 
                - CAST(rand(1)*60*86400 AS INT)
            )
        """)
    )
    
    # logged → +0–20 days
    .withColumn(
        "claim_logged_on",
        expr("""
            timestamp_seconds(
                unix_timestamp(incident_date)
                + CAST(rand(2)*20*86400 AS INT)
            )
        """)
    )
    
    # processed → +0–15 days
    .withColumn(
        "claim_processed_on",
        expr("""
            timestamp_seconds(
                unix_timestamp(Claim_Logged_On)
                + CAST(rand(3)*15*86400 AS INT)
            )
        """)
    )
)

In [0]:
display(t_claims_df)

**Dropping the rescued colum**

In [0]:
def drop_rescued_column(df):
    return df.drop("_rescued_data") if "_rescued_data" in df.columns else df

s_customers_df = drop_rescued_column(s_customers_df)
s_claims_df = drop_rescued_column(t_claims_df)
s_sales_df = drop_rescued_column(s_sales_df)
s_policy_df = drop_rescued_column(s_policy_df)
s_cars_df = drop_rescued_column(s_cars_df)

**Quarentine the invalid records**

In [0]:
from pyspark.sql.functions import col, when, count, lit, row_number
from pyspark.sql.window import Window

# Count nulls per row for each table 
def add_null_count(df):
    null_condition = sum([col(c).isNull().cast("int") for c in df.columns])
    return df.withColumn("null_count", null_condition)

s_customers_df = add_null_count(s_customers_df)
s_claims_df    = add_null_count(s_claims_df)
s_sales_df     = add_null_count(s_sales_df)
s_policy_df    = add_null_count(s_policy_df)
s_cars_df      = add_null_count(s_cars_df)


# Ranking the null for each table 
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

def add_rank(df, key_column):
    window_spec = Window.partitionBy(key_column).orderBy(col("null_count").asc())
    return df.withColumn("rank", row_number().over(window_spec))

customers_ranked_df = add_rank(s_customers_df, "customer_id")
claims_ranked_df    = add_rank(s_claims_df, "claim_id")
sales_ranked_df     = add_rank(s_sales_df, "sales_id")
policy_ranked_df    = add_rank(s_policy_df, "policy_id")
cars_ranked_df      = add_rank(s_cars_df, "car_id")

In [0]:
# Clean → best record AND no nulls #

from pyspark.sql.functions import col

def get_clean_df(df):
    return df.filter(
        (col("rank") == 1) & (col("null_count") == 0)
    ).drop("rank", "null_count")

customers_clean_df = get_clean_df(customers_ranked_df)
claims_clean_df    = get_clean_df(claims_ranked_df)
sales_clean_df     = get_clean_df(sales_ranked_df)
policy_clean_df    = get_clean_df(policy_ranked_df)
cars_clean_df      = get_clean_df(cars_ranked_df)

# Quarantine → duplicates OR null-containing records #
from pyspark.sql.functions import col, when, lit

def get_quarantine_df(df):
    return (
        df.withColumn(
            "error_type",
            when(col("null_count") > 0, lit("NULL_VALUES"))
            .when(col("rank") > 1, lit("DUPLICATE"))
        )
        .filter((col("rank") > 1) | (col("null_count") > 0))
        .drop("rank", "null_count")
    )

customers_quarantine_df = get_quarantine_df(customers_ranked_df)
claims_quarantine_df    = get_quarantine_df(claims_ranked_df)
sales_quarantine_df     = get_quarantine_df(sales_ranked_df)
policy_quarantine_df    = get_quarantine_df(policy_ranked_df)
cars_quarantine_df      = get_quarantine_df(cars_ranked_df)

In [0]:
def incremental_write_by_id(df, silver_path, key_column):

    try:
        existing_df = spark.table(silver_path).select(key_column)

        new_df = df.join(existing_df, on=key_column, how="left_anti")

        if new_df.limit(1).count() == 0:
            print(f"No new records for {silver_path}")
            return

        new_df.write.format("delta") \
            .mode("append") \
            .saveAsTable(silver_path)

        print(f"Inserted new records into {silver_path}")

    except:
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_path)

        print(f"Initial load completed for {silver_path}")

In [0]:
# customers split
incremental_write_by_id(customers_clean_df, customers_silver, "customer_id")
incremental_write_by_id(customers_quarantine_df, customers_quarantine_path, "customer_id")

# claims split
incremental_write_by_id(claims_clean_df, claims_silver, "claim_id")
incremental_write_by_id(claims_quarantine_df, claims_quarantine_path, "claim_id")

# sales split
incremental_write_by_id(sales_clean_df, sales_silver,"sales_id")
incremental_write_by_id(sales_quarantine_df, sales_quarantine_path, "sales_id")

# policy split
incremental_write_by_id(policy_clean_df, policy_silver, "policy_id")
incremental_write_by_id(policy_quarantine_df, policy_quarantine_path, "policy_number")

# cars split
incremental_write_by_id(cars_clean_df, cars_silver, "car_id")
incremental_write_by_id(cars_quarantine_df, cars_quarantine_path, "car_id")

In [0]:
show_df = spark.table(cars_silver)
display(show_df)

**Quality log table**

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS primeinsurance.silver.quality_log (
    issue_id STRING,
    table_name STRING,
    column_name STRING,
    issue_type STRING,
    severity STRING,
    affected_records BIGINT,
    affected_ratio DOUBLE,
    detected_at TIMESTAMP,
    details STRING,
    rule_name STRING,
    source_table STRING,
    batch_id STRING,
    source_file STRING,
    created_at TIMESTAMP,
    created_by STRING
)
USING DELTA
""")

In [0]:
import uuid
from pyspark.sql.functions import lit, current_timestamp

severity_map = {
    "null_violation": "High",
    "duplicate": "Medium",
    "invalid_format": "High",
    "schema_mismatch": "Critical",
    "range_error": "High"
}

def log_quality_issue(
    df,
    table_name,
    column_name,
    issue_type,
    severity,
    affected_records,
    total_records,
    rule_name,
    source_table=None,
    source_file=None,
    batch_id="batch_001",
    created_by="dq_pipeline_v1",
    details=None
):
    
    ratio = affected_records / total_records if total_records > 0 else 0.0

    log_df = spark.createDataFrame([(
        str(uuid.uuid4()),
        table_name,
        column_name,
        issue_type,
        severity,
        affected_records,
        ratio,
        None,
        details,
        rule_name,
        source_table,
        batch_id,
        source_file,
        None,
        created_by
    )], schema="""
        issue_id STRING,
        table_name STRING,
        column_name STRING,
        issue_type STRING,
        severity STRING,
        affected_records LONG,
        affected_ratio DOUBLE,
        detected_at TIMESTAMP,
        details STRING,
        rule_name STRING,
        source_table STRING,
        batch_id STRING,
        source_file STRING,
        created_at TIMESTAMP,
        created_by STRING
    """)

    log_df = log_df \
        .withColumn("detected_at", current_timestamp()) \
        .withColumn("created_at", current_timestamp())

    log_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("primeinsurance.silver.quality_log")

In [0]:
def process_dq_for_table(table_name, df, ranked_df, key_column):
    
    total_records = df.count()

    # NULL check
    null_count = df.filter("null_count > 0").count()
    if null_count > 0:
        log_quality_issue(
            table_name=table_name,
            column_name="ALL",
            issue_type="null_violation",
            affected_records=null_count,
            total_records=total_records,
            rule_name="dq_null_check"
        )

    # Duplicate check
    duplicate_count = ranked_df.filter("rank > 1").count()
    if duplicate_count > 0:
        log_quality_issue(
            table_name=table_name,
            column_name=key_column,
            issue_type="duplicate",
            affected_records=duplicate_count,
            total_records=total_records,
            rule_name="dq_duplicate_check"
        )

In [0]:
table_configs = [
    ("customers", s_customers_df, customers_ranked_df, "customer_id"),
    ("claims", s_claims_df, claims_ranked_df, "claimid"),
    ("sales", s_sales_df, sales_ranked_df, "sales_id"),
    ("policy", s_policy_df, policy_ranked_df, "policy_number"),
    ("cars", s_cars_df, cars_ranked_df, "car_id")
]
for table_name, df, ranked_df, key in table_configs:
    process_dq_for_table(table_name, df, ranked_df, key)

In [0]:
from pyspark.sql import Row

def get_claims_reject_count():
    try:
        return spark.table(claims_quarantine_path).count()
    except Exception:
        return 0

log_data = [
    Row(table="customers", issue="missing customer_id", count=customers_quarantine_df.count()),
    Row(table="claims", issue="invalid incident_date", count=get_claims_reject_count())
]

spark.createDataFrame(log_data).write.mode("overwrite").saveAsTable("primeinsurance.silver.quality_log")

In [0]:
i_df = spark.table("primeinsurance.silver.quality_log")
display(i_df)


In [0]:
df=spark.read.table (claims_silver)
display(df)

In [0]:
df=spark.read.table (claims_bronze)
display(df)